In [1]:
import torch, torch.onnx, tqdm, time, os, json, multiprocessing
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import ResidualRegression, DNNRegression
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_inputs = hyper_params['n_of_inputs']
n_outputs = hyper_params['n_of_outputs']
n_layers = hyper_params['n_of_layers']

In [6]:
model = DNNRegression(n_inputs, n_layers, n_outputs)
print(model)

DNNRegression(
  (input_layer): Linear(in_features=4, out_features=24, bias=True)
  (output_layer): Linear(in_features=24, out_features=6, bias=True)
  (active): ReLU()
  (model): Sequential(
    (0): Linear(in_features=4, out_features=24, bias=True)
    (1): ReLU()
    (2): Linear(in_features=24, out_features=24, bias=True)
    (3): ReLU()
    (4): Linear(in_features=24, out_features=24, bias=True)
    (5): ReLU()
    (6): Linear(in_features=24, out_features=24, bias=True)
    (7): ReLU()
    (8): Linear(in_features=24, out_features=6, bias=True)
  )
  (loss): MSELoss()
)


In [7]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=multiprocessing.cpu_count())

In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.001, patience=100)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='cuda', enable_progress_bar=True, max_epochs=10000)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type       | Params
--------------------------------------------
0 | input_layer  | Linear     | 120   
1 | output_layer | Linear     | 150   
2 | active       | ReLU       | 0     
3 | model        | Sequential | 2.1 K 
4 | loss         | MSELoss    | 0     
--------------------------------------------
2.1 K     Trainable params
0         Non-trainable params
2.1 K     Total params
0.008     Total estimated model params size (MB)
/home/jinbeom/workspace/estimate_crawler_crane_ground_pressure/venv/lib/python3.8/site-packages/pytorch_lightning/loops/fit_loop.py:280: PossibleUserWarning: The number of training batches (1) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Epoch 0: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s, v_num=61]

Metric train_loss improved. New best score: 13.940


Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  1.95it/s, v_num=61]

Metric train_loss improved by 1.971 >= min_delta = 0.001. New best score: 11.969


...

Epoch 747: 100%|██████████| 1/1 [00:00<00:00,  2.22it/s, v_num=61]

Metric train_loss improved by 0.001 >= min_delta = 0.001. New best score: 0.005


Epoch 832: 100%|██████████| 1/1 [00:00<00:00,  2.42it/s, v_num=61]

Metric train_loss improved by 0.001 >= min_delta = 0.001. New best score: 0.004


Epoch 891: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s, v_num=61]

In [12]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [10]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

In [11]:
torch.onnx.export(model, torch.zeros(4), './models/est_ground_pressure.onnx')